In [ ]:
#@title 1. 設定並同步專案程式碼
# --- 解說 ---
# 這個儲存格會完成兩項核心任務：
# 1. 設定 GitHub 專案的 URL 和分支名稱 (可在此處修改)。
# 2. 根據設定，自動從 GitHub 下載或更新最新的程式碼。
import os
from IPython import get_ipython

# --- 參數設定 ---
GITHUB_REPO_URL = "https://github.com/hsp1234-web/sp_lab.git" #@param {type:"string"}
GIT_BRANCH = "5.5" #@param {type:"string"}

print("✅ 參數設定完成！")
print(f"   - 儲存庫: {GITHUB_REPO_URL}")
print(f"   - 分支: {GIT_BRANCH}")

# --- 程式碼同步 ---
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
local_repo_path = os.path.join('/content', repo_name)
os.environ['LOCAL_REPO_PATH'] = local_repo_path

ipython = get_ipython()

if os.path.exists(local_repo_path):
    print(f"\n🌀 專案資料夾已存在，正在更新至 '{GIT_BRANCH}' 分支...")
    ipython.run_line_magic('cd', local_repo_path)
    ipython.run_line_magic('system', 'git fetch -q origin && git checkout -q {GIT_BRANCH} && git pull -q origin {GIT_BRANCH}')
    ipython.run_line_magic('cd', '/content')
else:
    print(f"\n✨ 專案資料夾不存在，正在從 '{GIT_BRANCH}' 分支下載...")
    ipython.run_line_magic('system', f'git clone -q --branch {GIT_BRANCH} {GITHUB_REPO_URL}')

print(f"✅ 程式碼同步完成！專案路徑: {local_repo_path}")


In [ ]:
#@title 2. 安裝 Python 相依套件
# --- 解說 ---
# 這個儲存格會使用 uv (一個高速的 Python 套件安裝工具) 來安裝 `requirements.txt` 中定義的所有套件。
#
# 備註：uv 會自動偵測已安裝的套件，只下載與安裝缺失或版本不符的部分。
# 因此，重複執行此儲存格是安全的，且不會浪費時間在重複安裝上。
import os, sys
from IPython import get_ipython

local_repo_path = os.environ['LOCAL_REPO_PATH']
requirements_path = os.path.join(local_repo_path, 'requirements.txt')

ipython = get_ipython()

if not os.path.exists(requirements_path):
    print(f"⚠️ 警告：在專案路徑中找不到 requirements.txt 檔案。")
else:
    # --- 釜底抽薪的解決方案 ---
    # 由於 Colab 環境預裝的 pandas 是用新版 numpy 編譯的，
    # 簡單降級 numpy 會導致致命的二進位衝突 (numpy.dtype size changed)。
    # 唯一的解決辦法是，在降級 numpy 之後，強制從原始碼重新編譯 pandas。
    print("📦 [階段 1/2] 正在安裝核心依賴...")
    ipython.run_line_magic('system', 'pip install -q uv && uv pip install -q -r {requirements_path}')
    print("✅ 核心依賴安裝完畢。")

    print("\n📦 [階段 2/2] 正在從原始碼重新編譯 pandas 以確保相容性...")
    print("   (此過程可能需要幾分鐘，請耐心等候...)")
    # --force-reinstall: 確保 pandas 被重新安裝
    # --no-binary pandas: 禁止使用預編譯的 wheel 檔案，強制從 sdist (原始碼) 編譯
    ipython.run_line_magic('system', 'uv pip install --force-reinstall --no-binary pandas pandas')
    print("✅ pandas 已成功重新編譯！環境已準備就緒。")



In [ ]:
#@title 3. 【一鍵執行】啟動主流程
# --- 解說 ---
# 這是專案的核心執行儲存格。
# 它會從專案的根目錄，呼叫我們設計的 `run.py` 腳本來啟動整個流程。
# `run.py` 會自動處理 Python 模組的路徑問題，並執行包含回測、視覺化的完整流程。
# 最終的權益曲線圖將會直接顯示在此儲存格的輸出中。
import os
from IPython import get_ipython
from IPython.display import Image, display

local_repo_path = os.environ['LOCAL_REPO_PATH']
run_script_path = os.path.join(local_repo_path, 'run.py')
equity_curve_path = os.path.join(local_repo_path, 'src', 'output', 'sp_equity_curve.jpg')

ipython = get_ipython()

if not os.path.exists(run_script_path):
    print(f"❌ 錯誤：找不到主執行檔 {run_script_path}")
else:
    # 執行 run.py 前，需確保當前目錄位於專案根目錄
    ipython.run_line_magic('cd', local_repo_path)

    # 使用 %run 來執行，這能確保腳本在當前的 IPython kernel 中執行
    ipython.run_line_magic('run', 'run.py')

    # 顯示最終產出的圖表
    if os.path.exists(equity_curve_path):
        print("\n--- 最終權益曲線圖 ---")
        display(Image(filename=equity_curve_path))
    else:
        print(f"⚠️ 警告：找不到預期的權益曲線圖檔案：{equity_curve_path}")

